# Le stranezze che vengono da li', e che si vedono ogni giorno

*Anatomia di un LLM* · capitolo 4, Dal testo ai numeri · script `02-03`, `scripts/02_03_patologie_tokenizer.py`

Se il modello riceve simboli gia' fusi e non lettere, alcune cose diventano
difficili per motivi che non hanno niente a che fare con l'intelligenza:
contare le lettere di una parola, distinguere una parola con lo spazio davanti
da una senza, fare i conti con numeri lunghi. Qui si misurano.

Si esegue con `python3 scripts/02_03_patologie_tokenizer.py`
e dura circa 5 secondi.

> **Per riottenere i numeri del libro** esegui le celle in ordine, dalla prima. Il seme si fissa una volta sola, in `prepara()`: rieseguire una cella a metà cambia i numeri casuali di tutte quelle dopo.

[Apri in Colab](https://colab.research.google.com/github/fcollova/llm-anatomy/blob/main/notebooks/02_03_patologie_tokenizer.ipynb) · la prima cella scarica il codice, se non lo trova accanto.

In [1]:
# Porta il notebook nella cartella che contiene llm_anatomy/, da dovunque
# sia stato aperto: gli script si eseguono da li', e cosi' anche le celle.
import os
import subprocess
import sys
from pathlib import Path

radice = next((p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "llm_anatomy").is_dir()), None)

if radice is None:
    # Il notebook e' stato aperto da solo, per esempio su Colab: il codice, il
    # testo e il modello addestrato si scaricano una volta, qui accanto.
    REPO, RAMO, CARTELLA = "fcollova/llm-anatomy", "main", ""
    copia = Path("anatomia-di-un-llm")
    if not copia.exists():
        esito = subprocess.run(
            ["git", "clone", "-q", "--depth", "1", "--branch", RAMO,
             f"https://github.com/{REPO}.git", str(copia)], capture_output=True)
        if esito.returncode != 0:
            raise SystemExit("Non riesco a scaricare il codice da github.com/"
                             + REPO + ": " + esito.stderr.decode()[-300:])
    radice = copia / CARTELLA

os.chdir(radice)
sys.path.insert(0, str(Path.cwd()))

In [2]:
from llm_anatomy.data import carica_testo
from llm_anatomy.tokenizer import addestra_bpe
from llm_anatomy.utils import dec, numero, prepara, tabella, titolo, verifica

prepara()

tok = addestra_bpe(carica_testo()[:400_000], vocab_size=1200)
print(f"  tokenizer BPE addestrato: {tok.get_vocab_size()} simboli")


def pezzi(s: str) -> list[str]:
    """Segmentazione, con i simboli non ASCII resi come sequenze di escape.

    Il marcatore di spazio del ByteLevel e' U+0120: stamparlo cosi' com'e'
    renderebbe l'output non componibile da LaTeX e illeggibile su terminali
    senza il font adatto.
    """
    return [p.encode("ascii", "backslashreplace").decode()
            for p in tok.encode(s).tokens]

[ambiente] torch 2.13.0+cpu  device=cpu  threads=8  seed=1337


  tokenizer BPE addestrato: 1200 simboli


In [3]:
titolo("Perche' sbaglia a contare le lettere")
for parola, lettera in (("prezzemolo", "z"), ("sottilissime", "s"),
                        ("mescolatelo", "e")):
    p = pezzi(parola)
    print(f"  {parola!r:<16} -> {p}")
    quante = parola.count(lettera)
    print(f"  {'':<16}    {quante} lettere {lettera!r} sparse in {len(p)} "
          f"token, mescolate ad altre")
print("  chiedere quante lettere ci sono significa chiedere di contare dentro")
print("  simboli che il modello riceve gia' fusi")


Perche' sbaglia a contare le lettere
------------------------------------
  'prezzemolo'     -> ['pre', 'zzemolo']
                      2 lettere 'z' sparse in 2 token, mescolate ad altre
  'sottilissime'   -> ['so', 'tti', 'li', 'ssi', 'me']
                      3 lettere 's' sparse in 5 token, mescolate ad altre
  'mescolatelo'    -> ['m', 'es', 'colate', 'lo']
                      2 lettere 'e' sparse in 4 token, mescolate ad altre
  chiedere quante lettere ci sono significa chiedere di contare dentro
  simboli che il modello riceve gia' fusi


In [4]:
titolo("Lo spazio davanti cambia tutto")
righe = []
for parola in ("Burro", "burro"):
    for prefisso, nome in ((" ", "con spazio"), ("", "senza spazio")):
        p = pezzi(prefisso + parola)
        righe.append([f"{prefisso + parola!r}", nome, len(p), " ".join(p)])
tabella(["stringa", "forma", "n", "token"], righe)
print("  la stessa parola ha identita' diverse a seconda del contesto:")
print("  e' la ragione per cui i prompt che finiscono con uno spazio si")
print("  comportano in modo diverso da quelli che non ce l'hanno")


Lo spazio davanti cambia tutto
------------------------------
stringa          forma  n        token
--------  ------------  -  -----------
' Burro'    con spazio  1  \u0120Burro
'Burro'   senza spazio  2       B urro
' burro'    con spazio  1  \u0120burro
'burro'   senza spazio  1        burro
  la stessa parola ha identita' diverse a seconda del contesto:
  e' la ragione per cui i prompt che finiscono con uno spazio si
  comportano in modo diverso da quelli che non ce l'hanno


In [5]:
titolo("I numeri, che nessuno taglia bene")
righe = []
for n in ("7", "42", "365", "1265", "1000000", "3.14159"):
    p = pezzi(n)
    righe.append([n, len(p), " ".join(repr(x) for x in p)])
tabella(["numero", "token", "segmentazione"], righe)
print("  cifre raggruppate in modo arbitrario: la posizione delle unita' non")
print("  e' una proprieta' visibile al modello, e l'aritmetica in colonna")
print("  richiede allora di ricostruirla dai pezzi")


I numeri, che nessuno taglia bene
---------------------------------
numero   token                segmentazione
-------  -----  ---------------------------
7            1                          '7'
42           2                      '4' '2'
365          3                  '3' '6' '5'
1265         4              '1' '2' '6' '5'
1000000      4           '1' '00' '00' '00'
3.14159      7  '3' '.' '1' '4' '1' '5' '9'
  cifre raggruppate in modo arbitrario: la posizione delle unita' non
  e' una proprieta' visibile al modello, e l'aritmetica in colonna
  richiede allora di ricostruirla dai pezzi


In [6]:
titolo("Costa di piu' scrivere in altre lingue")
campioni = {
    "italiano": "Mettete al fuoco il burro e fate bollire il latte adagio.",
    "inglese": "Put the butter on the fire and let the milk boil slowly.",
    "tedesco": "Die Butter aufs Feuer stellen und die Milch langsam kochen.",
}
righe = []
for lingua, frase in campioni.items():
    n_tok = len(pezzi(frase))
    n_parole = len(frase.split())
    righe.append([lingua, len(frase), n_tok, n_parole, dec(n_tok / n_parole)])
tabella(["lingua", "caratteri", "token", "parole", "token per parola"], righe)
inglese = len(pezzi(campioni["inglese"])) / len(campioni["inglese"].split())
italiano = len(pezzi(campioni["italiano"])) / len(campioni["italiano"].split())
print(f"  l'inglese costa {dec(inglese / italiano)} volte l'italiano a "
      "parita' di")
print("  contenuto: stessa finestra di contesto, meno testo dentro, e un")
print("  conto a token che cresce nella stessa proporzione")
verifica(inglese > italiano,
         "il costo per parola e' maggiore nella lingua che il corpus non "
         "contiene")


Costa di piu' scrivere in altre lingue
--------------------------------------
lingua    caratteri  token  parole  token per parola
--------  ---------  -----  ------  ----------------
italiano         57     15      11              1,36
inglese          56     31      12              2,58
tedesco          59     36      10              3,60
  l'inglese costa 1,89 volte l'italiano a parita' di
  contenuto: stessa finestra di contesto, meno testo dentro, e un
  conto a token che cresce nella stessa proporzione
  [verifica superata] il costo per parola e' maggiore nella lingua che il
                      corpus non contiene


In [7]:
titolo("Una causa sola")
print("  il modello riceve identificatori interi, non testo. Ogni patologia")
print("  qui sopra e' la stessa osservazione vista da un lato diverso: cio'")
print("  che non e' un token non e' un'unita' su cui il modello possa operare")


Una causa sola
--------------
  il modello riceve identificatori interi, non testo. Ogni patologia
  qui sopra e' la stessa osservazione vista da un lato diverso: cio'
  che non e' un token non e' un'unita' su cui il modello possa operare


## Da provare

aggiungi una parola italiana lunga e conta i suoi pezzi.